In [6]:
# !pip install opencv-python
# !pip install torch torchvision torchaudio albumentations torchattacks tqdm
# !pip install tf-keras

In [7]:
############################## 1. Haar Cascade, Siamese, FGSM (in a loop)
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import re
# from torchattacks import FGSM

# ==============================
# Device Setup
# ==============================
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================
# Paths
# ==============================
base_path = './AdvLFW/images'

# ==============================
# Haar Cascade Face Detection
# Haar Cascade Face Detection is a fast, classical computer vision method that uses Haar-like rectangular features 
# and a cascade of classifiers to detect faces in images. 
# It leverages integral images for rapid feature computation.
# ==============================
# Load Haar cascade once
haar_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_cascade_path)

def detect_face_haar(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(faces) > 0:
        x, y, w, h = faces[0]
        face = image[y:y+h, x:x+w]
        face = cv2.resize(face, (112, 112))
        return face
    else:
        return np.zeros((112, 112, 3), dtype=np.uint8)

# ==============================
# Dataset
# ==============================
class SiameseFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.pairs = []
        self.transform = transform

        # Walk through all subdirectories
        for root, _, _ in os.walk(root_dir):
            match = re.search(r'label_(\d+)', root)
            if not match:
                continue
            label = int(match.group(1))
            im0 = next((f for f in os.listdir(root) if f.startswith("im_0_")), None)
            im1 = next((f for f in os.listdir(root) if f.startswith("im_1_")), None)
            if im0 and im1:
                img0_path = os.path.join(root, im0)
                img1_path = os.path.join(root, im1)
                self.pairs.append((img0_path, img1_path, label))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img0_path, img1_path, label = self.pairs[idx]
        img0 = cv2.imread(img0_path)
        img1 = cv2.imread(img1_path)
        img0 = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        img0 = detect_face_haar(img0)
        img1 = detect_face_haar(img1)
        if self.transform:
            img0 = self.transform(image=img0)['image']
            img1 = self.transform(image=img1)['image']
        return img0.to(torch.float32), img1.to(torch.float32), torch.tensor(label, dtype=torch.float32)

# ==============================
# Transforms
# ==============================
transform = A.Compose([
    A.Resize(112, 112),
    A.Normalize(),
    ToTensorV2()
])

# ==============================
# Siamese Network
# "Siamese Network instead of classifying a face image to one of the classes, takes a reference image of the person as input 
# and calculates a similarity score, which tells us whether the two input images represent the same person."
# ==============================
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        with torch.no_grad():
            dummy = self.backbone(torch.zeros(1, 3, 112, 112))
            self.flattened = dummy.view(1, -1).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(self.flattened, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward_once(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return F.normalize(self.fc(x), p=2, dim=1)  # L2 normalize for cosine similarity

    def forward(self, x1, x2):
        return self.forward_once(x1), self.forward_once(x2)

# ==============================
# Contrastive Loss
# Contrastive loss is commonly used in Siamese networks and works by minimizing the distance between positive pairs 
# and maximizing the distance between negative pairs, typically using Euclidean or cosine distance.
# ==============================
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, label):
        cosine = F.cosine_similarity(out1, out2)
        dist = 1 - cosine  # distance for cosine similarity
        loss = label * dist.pow(2) + (1 - label) * F.relu(self.margin - dist).pow(2)
        return loss.mean()

# ==============================
# Dataset, Model, Loss, Optimizer
# ==============================
dataset = SiameseFaceDataset(base_path, transform=transform)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(dataset, batch_size=1, shuffle=False)

model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# fgsm = FGSM(model, eps=0.01)

# ==============================
# Training Loop (with FGSM)
# ==============================
def train(model, loader, criterion, optimizer, epochs=10, use_fgsm=True):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for img1, img2, labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)

            # Optionally apply FGSM adversarial attack to img1
            if use_fgsm:
                img1.requires_grad = True
                out1, out2 = model(img1, img2)
                loss = criterion(out1, out2, labels)
                model.zero_grad()
                loss.backward()
                img1_adv = img1 + 0.01 * img1.grad.sign()
                img1_adv = torch.clamp(img1_adv, 0, 1).detach()
                out1, out2 = model(img1_adv, img2)
                loss = criterion(out1, out2, labels)
            else:
                out1, out2 = model(img1, img2)
                loss = criterion(out1, out2, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] Avg Loss: {total_loss / len(loader):.4f}")

# ==============================
# Evaluation with Cosine Similarity
# ==============================
def evaluate(model, loader, threshold=0.819):
    model.eval()
    y_true, y_pred, sims = [], [], []
    with torch.no_grad():
        for img1, img2, label in loader:
            img1, img2 = img1.to(device), img2.to(device)
            out1, out2 = model(img1, img2)
            sim = F.cosine_similarity(out1, out2).item()
            pred = 1 if sim > threshold else 0
            y_true.append(int(label.item()))
            y_pred.append(pred)
            sims.append(sim)
    acc = accuracy_score(y_true, y_pred)
    try: roc_auc = roc_auc_score(y_true, sims)
    except: roc_auc = None
    print("\nEvaluation:")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ==============================
# Run Training and Evaluation
# ==============================
train(model, train_loader, criterion, optimizer, epochs=10, use_fgsm=True)
evaluate(model, test_loader, threshold=0.819)


Using device: mps


Epoch 1/10: 100%|█████████████████████████████| 750/750 [01:37<00:00,  7.70it/s]


Epoch [1/10] Avg Loss: 0.0666


Epoch 2/10: 100%|█████████████████████████████| 750/750 [01:34<00:00,  7.98it/s]


Epoch [2/10] Avg Loss: 0.0631


...

Epoch [8/10] Avg Loss: 0.0328


Epoch 9/10: 100%|█████████████████████████████| 750/750 [01:35<00:00,  7.88it/s]


Epoch [9/10] Avg Loss: 0.0194


Epoch 10/10: 100%|████████████████████████████| 750/750 [01:35<00:00,  7.88it/s]


Epoch [10/10] Avg Loss: 0.0093

Evaluation:
Accuracy: 0.8360
ROC AUC: 0.9214
Confusion Matrix:
 [[2710  290]
 [ 694 2306]]


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_with_metrics(model, loader, threshold=0.819):
    model.eval()
    y_true, y_pred, sims = [], [], []
    with torch.no_grad():
        for img1, img2, label in loader:
            img1, img2 = img1.to(device), img2.to(device)
            out1, out2 = model(img1, img2)
            sim = F.cosine_similarity(out1, out2).item()
            pred = 1 if sim > threshold else 0
            y_true.append(int(label.item()))
            y_pred.append(pred)
            sims.append(sim)
    acc = accuracy_score(y_true, y_pred)
    try:
        roc_auc = roc_auc_score(y_true, sims)
    except:
        roc_auc = None
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print("\nEvaluation with additional metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

evaluate_with_metrics(model, test_loader, threshold=0.819)


Evaluation with additional metrics:
Accuracy: 0.8360
ROC AUC: 0.9214
Precision: 0.8883
Recall: 0.7687
F1 Score: 0.8242
Confusion Matrix:
 [[2710  290]
 [ 694 2306]]
